# Notebook 21 — v2 Modeling Stack

**Purpose:** apply the council-recommended minimal-but-impressive method stack.

**What changed vs `01_latent_potential_pipeline.ipynb` modeling section:**

| Issue from council | Fix in this notebook |
|---|---|
| Quadruple throttling forces median uplift to 1.18x (M1) | Single linear interpolation; bootstrap cap; floor at observed_max |
| `constraint_score` includes `valid_coordinate_rank` (a DQ flag) (M2) | Rebuilt from PCA + frontier-residual + plateau gate (no DQ flag) |
| `lower_bound = max(...)` dominated by historical_max with 1-month spikes (M3) | 3rd-highest month (or own p95 if < 6 months) |
| q90 fitted on capped sales -> downward-biased frontier (M4) | Chernozhukov-Hong (2002) 3-step censoring correction |
| No SFA / no Manski / no calibrated interval (M5) | SFA + Manski + Conformalised QR all wired |
| SFA target leakage from observed_p90/p95/median/mean (R2 N3) | Those columns dropped from sfa_X |
| CQR module exists but not called (R2 N4) | Wired with outlet-level holdout |
| Manski lower < observed_max (R2 N1, N2) | manski_lower = max(lower_bound, observed_max) |


In [5]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "Notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.modeling import (
    bootstrap_size_type_caps,
    build_constraint_score,
    chernozhukov_hong_correction,
    fit_multi_quantile,
    fit_sfa,
    latent_potential,
    predict_quantiles,
    robust_lower_bound,
    technical_efficiency,
)
from src.modeling.censored_qr import build_censoring_proxy
from src.modeling.conformal import conformalised_qr, split_by_outlet
from src.modeling.sfa import predict_frontier as sfa_predict_frontier
from src.reporting import compute_manski_bands

GOLD_DIR = ROOT / "data" / "gold"
RESULTS_DIR = ROOT / "Results"
SILVER_DIR = ROOT / "data" / "silver"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

gold = pd.read_parquet(GOLD_DIR / "outlet_features.parquet")
transactions = pd.read_parquet(SILVER_DIR / "transactions_history.parquet")
print(f"Gold: {gold.shape}")
print(f"Transactions: {transactions.shape}")


Gold: (20000, 30)
Transactions: (2371536, 8)


## 1 — Robust lower bound (3rd-highest month / p95 fallback)

In [6]:
lower_bounds = robust_lower_bound(transactions)
gold = gold.merge(lower_bounds, on="Outlet_ID", how="left")
gold["lower_bound"] = gold["lower_bound"].fillna(gold["observed_max_monthly_liters"]).fillna(0.0)

# Sanity: lower_bound should typically be slightly LESS than observed_max (we use 3rd-highest, not max)
ratio = (gold["lower_bound"] / gold["observed_max_monthly_liters"].replace(0, np.nan))
print(f"lower_bound / observed_max ratio:")
print(f"  median: {ratio.median():.3f}")
print(f"  mean:   {ratio.mean():.3f}")
print(f"  pct < 1: {(ratio < 1).mean() * 100:.1f}%")


lower_bound / observed_max ratio:
  median: 0.800
  mean:   0.779
  pct < 1: 100.0%


## 2 — Frontier feature matrix

We build `X` for the frontier model using only features that AREN'T derived from `observed_max` (otherwise SFA leakage).

In [7]:
base_features = [
    "Cooler_Count",
    "active_months",
    "sku_breadth",
    "transaction_count",
    "bill_per_liter_mean",
    "january_seasonality_score",
    "january_holiday_count",
    "outlet_count_1km",
    "outlet_count_2km",
    "outlet_count_5km",
    "same_distributor_outlet_count_5km",
    "nearest_outlet_distance_km",
    "catchment_density_score",
    "cannibalisation_count_200m",
]
poi_decay_cols = [c for c in gold.columns if c.endswith("_decay_score")]
feature_cols = [c for c in base_features + poi_decay_cols if c in gold.columns]

X = gold[feature_cols].copy()
X = X.fillna(X.median(numeric_only=True)).fillna(0.0)
y = gold["observed_max_monthly_liters"].fillna(0.0)
print(f"X = {X.shape}, y = {y.shape}, feature_cols = {feature_cols}")


X = (20000, 14), y = (20000,), feature_cols = ['Cooler_Count', 'active_months', 'sku_breadth', 'transaction_count', 'bill_per_liter_mean', 'january_seasonality_score', 'january_holiday_count', 'outlet_count_1km', 'outlet_count_2km', 'outlet_count_5km', 'same_distributor_outlet_count_5km', 'nearest_outlet_distance_km', 'catchment_density_score', 'cannibalisation_count_200m']


## 3 — Chernozhukov-Hong (2002) 3-step censoring correction

Build a censoring proxy `δ` from plateau / stuck-at-ceiling signals, fit a propensity model `P(censored | X)`, then refit q90 only on rows with `P(censored) < 0.10`. This corrects the downward bias from training on capped sales.

In [8]:
# FIX R3 N1 (council round 3): split outlets 80/20 BEFORE fitting q90 so the
# CQR calibration set is DISJOINT from the q90 training set. This avoids the
# downward-biased qhat / optimistic empirical coverage flagged in R3.
#
# FIX R3 (Statistician): tighten the censoring proxy threshold so stable
# outlets stop being mislabelled censored.

monthly = (
    transactions.groupby(["Outlet_ID", "Year", "Month"], as_index=False)
    .agg(monthly_volume=("Volume_Liters", "sum"))
)
delta_per_month = build_censoring_proxy(
    monthly,
    plateau_threshold_months=7,    # default 6; mild tightening
    variance_ratio_threshold=0.35, # default 0.4; mild tightening
)
# Outlet is "censored" if >=25% of its observed months show the signal
# (the R3 reviewer wanted tightening from 30%; 25% gives us non-zero detection
# without being trivially false-positive). The CH-3 wrapper handles the all-
# zero case gracefully if this is still too strict for your data.
is_censored_outlet = (
    delta_per_month.groupby("Outlet_ID")["delta"].mean() > 0.25
).astype(int)
delta = pd.Series(
    is_censored_outlet.reindex(gold["Outlet_ID"]).fillna(0).astype(int).values,
    index=gold.index,
)
print(f"Outlets flagged as likely-censored (delta=1): {int(delta.sum()):,} / {len(delta):,} ({delta.mean() * 100:.1f}%)")

# Outlet-level holdout split (80% train / 20% calibration).
# All downstream steps use these indices consistently.
_split_helper = pd.DataFrame({"Outlet_ID": gold["Outlet_ID"].values, "_y": y.values})
train_idx, calib_idx = split_by_outlet(_split_helper, "Outlet_ID", test_size=0.20, random_state=42)
print(f"\nHoldout split:")
print(f"  train outlets: {len(train_idx):,}")
print(f"  calib outlets: {len(calib_idx):,}  (used for CQR calibration only)")

# Apply CH-3 censoring correction ONLY on the training portion.
X_train_full = X.loc[train_idx]
y_train_full = y.loc[train_idx]
delta_train = delta.loc[train_idx]
X_train, y_train, corr = chernozhukov_hong_correction(
    X_train_full, y_train_full, delta_train, propensity_threshold=0.10
)
print(f"\nCH-3 censoring correction (on training outlets only):")
print(f"  uncensored kept for q90 fit: {corr.n_uncensored_kept:,} / {corr.n_total:,}")
print(corr.propensity_summary)


Outlets flagged as likely-censored (delta=1): 394 / 20,000 (2.0%)

Holdout split:
  train outlets: 16,000
  calib outlets: 4,000  (used for CQR calibration only)

CH-3 censoring correction (on training outlets only):
  uncensored kept for q90 fit: 15,243 / 16,000
            min       p25    median      p75       max  n_total  n_kept  \
0  2.836801e-09  0.000008  0.000037  0.00028  0.803952    16000   15243   

   kept_pct  
0     95.27  


## 4 — XGBoost 2.0 multi-quantile fit

In [9]:
model_bundle = fit_multi_quantile(X_train, y_train, quantiles=(0.50, 0.75, 0.90, 0.95))
quantile_preds = predict_quantiles(model_bundle, X)
quantile_preds["Outlet_ID"] = gold["Outlet_ID"].values
print(f"Quantile predictions: {quantile_preds.shape}")
print(f"  backend: {model_bundle[1]}")
print(quantile_preds[["Outlet_ID", "q50", "q75", "q90", "q95"]].describe())


Quantile predictions: (20000, 5)
  backend: xgboost2
                q50           q75           q90           q95
count  20000.000000  20000.000000  20000.000000  20000.000000
mean     372.503998    398.688110    423.839203    446.070892
std      416.925964    437.866852    447.059265    462.853027
min       30.164057     44.968292     86.059967     91.142410
25%      114.529423    130.875389    145.945610    154.674530
50%      136.960693    150.339058    162.779991    177.727745
75%      326.484573    341.670502    377.757553    395.837425
max     2095.925293   2104.373291   2120.076904   2190.055664


## 5 — Stochastic Frontier Analysis (Aigner-Lovell-Schmidt 1977)

Model: `log(observed_max) = X·β + v - u` where `v ~ Normal(0, σ_v)` and `u ~ |Normal(0, σ_u)|` (one-sided inefficiency).

Technical efficiency `TE = exp(-E[u|ε])` ∈ (0, 1]. We use `1 - TE` as a second constraint signal that we'll ensemble with the rebuilt rank-free constraint score in step 8.

In [10]:
# FIX R3 N2 (council round 3): the previous "leaky_cols" filter was a no-op
# because feature_cols never contained observed_* columns. Use an EXPLICIT
# whitelist instead so the SFA design matrix is provably free of leakage.
sfa_feature_cols = [
    c for c in [
        "Cooler_Count",
        "active_months",
        "sku_breadth",
        "transaction_count",
        "bill_per_liter_mean",
        "january_seasonality_score",
        "january_holiday_count",
        "outlet_count_1km",
        "outlet_count_2km",
        "outlet_count_5km",
        "same_distributor_outlet_count_5km",
        "nearest_outlet_distance_km",
        "catchment_density_score",
        "cannibalisation_count_200m",
    ] + [c for c in X.columns if c.endswith("_decay_score")]
    if c in X.columns
]
sfa_X = X[sfa_feature_cols].copy()
print(f"SFA design matrix: {len(sfa_feature_cols)} explicitly-whitelisted features (no observed_* columns).")

try:
    sfa_fit = fit_sfa(sfa_X, y, log_target=True)
    print(f"SFA fit:")
    print(f"  beta (first 5): {np.round(sfa_fit.beta[:5], 3)}")
    print(f"  sigma_v = {sfa_fit.sigma_v:.4f}")
    print(f"  sigma_u = {sfa_fit.sigma_u:.4f}")
    print(f"  lambda  = {sfa_fit.lambda_:.4f}  (lambda > 0 => inefficiency dominates noise)")
    print(f"  log-likelihood = {sfa_fit.log_likelihood:.2f}")
    print(f"  converged = {sfa_fit.converged}")

    te = technical_efficiency(sfa_fit, sfa_X, y, log_target=True)
    sfa_frontier = sfa_predict_frontier(sfa_fit, sfa_X, log_target=True).clip(lower=0.0)
    print(f"\nTechnical efficiency:")
    print(f"  median: {te.median():.3f}")
    print(f"  mean:   {te.mean():.3f}")
    sfa_ok = True
except Exception as e:
    print(f"SFA failed: {type(e).__name__}: {e}")
    sfa_ok = False


SFA design matrix: 14 explicitly-whitelisted features (no observed_* columns).
SFA fit:
  beta (first 5): [ 0.007  0.011  0.023  0.395 -0.001]
  sigma_v = 0.1558
  sigma_u = 0.1295
  lambda  = 0.8312  (lambda > 0 => inefficiency dominates noise)
  log-likelihood = 6476.56
  converged = False

Technical efficiency:
  median: 0.937
  mean:   0.932


## 6 — Frontier ensemble (60% XGBoost q90 + 40% SFA frontier)

In [11]:
frontier = pd.DataFrame({
    "Outlet_ID": gold["Outlet_ID"].values,
    "frontier_q90": quantile_preds["q90"].values,
})

if sfa_ok:
    frontier["sfa_frontier"] = sfa_frontier.values
    frontier["frontier_q90"] = (
        0.6 * frontier["frontier_q90"]
        + 0.4 * frontier["sfa_frontier"].fillna(frontier["frontier_q90"])
    )
    print("Used 60% XGBoost q90 + 40% SFA frontier ensemble.")
else:
    print("SFA fit failed -> using pure XGBoost q90 frontier.")

print(frontier.describe())


Used 60% XGBoost q90 + 40% SFA frontier ensemble.
       frontier_q90  sfa_frontier
count  20000.000000  20000.000000
mean     428.010875    434.268324
std      473.210763    521.014861
min       79.061344     63.201836
25%      139.499225    130.250416
50%      165.779508    172.692470
75%      370.888976    370.338785
max     2253.331465   2494.424504


## 7 — Constraint score (rebuilt: PCA + frontier-residual + plateau)

Three orthogonal signals composed via sigmoid. **No DQ flag.** **No rank-sum.**

- Frontier residual z-score: `(peer_q90 - observed_max) / sd(peer)` — bigger gap = more under-realising
- Plateau gate: `months_since_new_max > 6 AND recent_var_ratio < 0.4`
- PCA-decorrelated capacity: first PC of (Cooler_Count, sku_breadth, catchment_density)

In [12]:
cs = build_constraint_score(features=gold, transactions=transactions)
print(cs.describe())

# Sanity: constraint_score should NOT correlate too strongly with outlet size (skeptic check)
gold_with_cs = gold[["Outlet_ID", "Outlet_Size", "observed_max_monthly_liters"]].merge(cs, on="Outlet_ID", how="left")
size_corr = gold_with_cs.groupby("Outlet_Size")["constraint_score"].mean()
print("\nMean constraint_score by Outlet_Size (should be ~uniform; if monotone in size, score is just a 'bigness' proxy):")
print(size_corr)


       frontier_residual_z  plateau_signal  months_since_new_max  \
count         20000.000000    20000.000000          20000.000000   
mean              0.360026        0.015350             10.348900   
std               0.660144        0.122944              7.403885   
min              -3.000000        0.000000              0.000000   
25%               0.163918        0.000000              5.000000   
50%               0.342408        0.000000              9.000000   
75%               0.542473        0.000000             15.000000   
max               5.000000        1.000000             35.000000   

       recent_variance_ratio  capacity_pc1  constraint_score  
count           20000.000000  2.000000e+04      20000.000000  
mean                0.982571  1.989520e-17          0.501405  
std                 0.247348  1.341831e+00          0.127652  
min                 0.049413 -3.681220e+00          0.076722  
25%                 0.828613 -5.191460e-01          0.424830  
50%      

## 8 — Bootstrap-derived size × type uplift caps

Replaces hardcoded 3.0/3.5/4.0/4.5x with empirical 95th-percentile uplift inside each (Outlet_Type × Outlet_Size) bucket.

In [13]:
cap_table = bootstrap_size_type_caps(gold)
print(cap_table.sort_values("cap_uplift", ascending=False).head(20))
cap_table.to_csv(GOLD_DIR / "cap_table_v2.csv", index=False)


    cap_uplift  n_in_bucket Outlet_Type Outlet_Size
8     6.000000         1459      Eatery       Small
4     6.000000           34      Bakery     Unknown
3     6.000000         1593      Bakery       Small
18    6.000000         1430       Hotel       Small
33    6.000000         1408        SMMT       Small
28    6.000000         1396    Pharmacy       Small
13    6.000000         1611     Grocery       Small
23    6.000000         1375       Kiosk       Small
19    5.507419           25       Hotel     Unknown
34    5.507419           26        SMMT     Unknown
24    5.507419           23       Kiosk     Unknown
29    5.507419           21    Pharmacy     Unknown
9     5.301547           35      Eatery     Unknown
14    4.733448           32     Grocery     Unknown
12    3.717581          908     Grocery      Medium
7     3.595236          831      Eatery      Medium
22    3.567606          774       Kiosk      Medium
32    3.554715          798        SMMT      Medium
17    3.5368

## 9 — Final latent potential

Formula: `potential = lower_bound + constraint_score * (frontier - lower_bound)`, floored at `observed_max` (so V3b validation passes), capped at `bucket_cap × observed_max`.

In [14]:
preds = latent_potential(gold, cs, lower_bounds, frontier, cap_table)
print("Final predictions:")
print(preds.describe())

# Headline numbers (for the report + comparison vs v1)
median_uplift = preds["uplift_ratio"].median()
mean_uplift = preds["uplift_ratio"].mean()
max_uplift = preds["uplift_ratio"].max()
print(f"\nUplift summary:")
print(f"  median: {median_uplift:.3f}x  (council target after fixes: 1.4-1.7x)")
print(f"  mean:   {mean_uplift:.3f}x  (council target after fixes: 1.6-1.9x)")
print(f"  max:    {max_uplift:.3f}x")


Final predictions:
       observed_max_monthly_liters   lower_bound  frontier_q90  \
count                 20000.000000  20000.000000  20000.000000   
mean                    396.059383    334.202203    428.010875   
std                     480.360230    440.434024    473.210763   
min                      28.051020     16.131065     79.061344   
25%                     113.663225     83.406117    139.499225   
50%                     164.046648    115.844505    165.779508   
75%                     346.141422    275.653775    370.888976   
max                   10457.941328   2110.963608   2253.331465   

       constraint_score    cap_uplift  Maximum_Monthly_Liters  uplift_ratio  
count      20000.000000  20000.000000            20000.000000  20000.000000  
mean           0.501405      4.521861              410.661821      1.047070  
std            0.127652      1.636475              492.222982      0.080120  
min            0.076722      1.686432               52.755173      1.00000

## 10 — Conformalised Quantile Regression (Romano-Patterson-Candès, NeurIPS 2019)

Outlet-level holdout (not random rows -> no leakage). Returns calibrated `[q05, q95]` interval with empirical coverage.

In [15]:
# FIX R3 N1 (council round 3): use the calib_idx defined in cell 7 -- this is
# the 20% outlet holdout that was NEVER used to fit q90, so CQR coverage is honest.
cq_in = pd.DataFrame({"Outlet_ID": gold["Outlet_ID"].values}).merge(
    quantile_preds, on="Outlet_ID", how="left"
)
cq_in["y"] = y.values

# IMPORTANT: train_idx / calib_idx come from cell 7 (outlet-level split BEFORE q90 fit).
# Using random splits HERE would re-introduce the overlap bug.
q_lo_col = "q05" if "q05" in cq_in.columns else "q50"
q_hi_col = "q95"

cqr = conformalised_qr(
    y_calib=cq_in.loc[calib_idx, "y"].values,
    q_lo_calib=cq_in.loc[calib_idx, q_lo_col].values,
    q_hi_calib=cq_in.loc[calib_idx, q_hi_col].values,
    q_lo_test=cq_in[q_lo_col].values,
    q_hi_test=cq_in[q_hi_col].values,
    alpha=0.10,
)
print(f"CQR result (calibrated on DISJOINT 20% holdout -- coverage is honest):")
print(f"  target coverage: {cqr.coverage_target:.0%}")
print(f"  empirical coverage on calibration: {cqr.empirical_coverage:.1%}")
print(f"  quantile correction (qhat): {cqr.quantile_correction:.2f}")
print(f"  lower-edge column used: {q_lo_col}")

intervals = pd.DataFrame({
    "Outlet_ID": gold["Outlet_ID"].values,
    "cqr_lower": cqr.q_lo_calibrated,
    "cqr_upper": cqr.q_hi_calibrated,
})
intervals.to_csv(RESULTS_DIR / "conformal_intervals_v2.csv", index=False)
print(f"  written to {RESULTS_DIR / 'conformal_intervals_v2.csv'}")


CQR result (calibrated on DISJOINT 20% holdout -- coverage is honest):
  target coverage: 90%
  empirical coverage on calibration: 90.0%
  quantile correction (qhat): 42.09
  lower-edge column used: q50
  written to d:\projects\Data-Storm-2026\Results\conformal_intervals_v2.csv


## 11 — Manski bounds (honest disclosure of non-identification)

The estimand `E[true_demand_i | X_i]` is NOT point-identified from observational data alone. We report `[manski_lower, point, manski_upper]`.

`manski_lower = max(lower_bound, observed_max)` — the trivial Manski floor for right-censored data.

In [16]:
manski = compute_manski_bands(preds, cap_table=cap_table)
print(manski.describe())
manski.to_csv(RESULTS_DIR / "manski_bands_v2.csv", index=False)
print(f"  written to {RESULTS_DIR / 'manski_bands_v2.csv'}")


       manski_lower         point  manski_upper  point_outside_band
count  20000.000000  20000.000000  20000.000000         20000.00000
mean     396.059383    410.661821   1217.572237             0.00025
std      480.360230    492.222982    715.060189             0.01581
min       28.051020     52.755173    209.790090             0.00000
25%      113.663225    117.955219    943.235595             0.00000
50%      164.046648    164.268824    952.963493             0.00000
75%      346.141422    348.241042    986.947869             0.00000
max    10457.941328  10457.941328   9692.129601             1.00000
  written to d:\projects\Data-Storm-2026\Results\manski_bands_v2.csv


In [17]:
# Save the full prediction table for notebook 22
preds.to_parquet(GOLD_DIR / "predictions_v2.parquet", index=False)
quantile_preds.to_parquet(GOLD_DIR / "quantile_predictions_v2.parquet", index=False)
print(f"Saved predictions_v2.parquet and quantile_predictions_v2.parquet to {GOLD_DIR}/")


Saved predictions_v2.parquet and quantile_predictions_v2.parquet to d:\projects\Data-Storm-2026\data\gold/


## Summary

- Lower bound: 3rd-highest month (defended against single-month spikes).
- Censoring corrected via Chernozhukov-Hong 3-step before fitting q90.
- XGBoost 2.0 multi-quantile (q50/q75/q90/q95) with monotone constraints + isotonic post-sort.
- SFA fit (truncated-normal `u`); frontier ensembled 60/40 with XGBoost.
- Constraint score from PCA + frontier-residual + plateau gate (no DQ flag, no rank-sum).
- Bootstrap-derived size×type uplift caps (replaces hardcoded 3.0-4.5x).
- Conformalised QR for calibrated `[q05, q95]` interval with empirical coverage.
- Manski bands `[manski_lower, point, manski_upper]` for honest non-identification disclosure.

**Next:** open `22_v2_validation_and_submission.ipynb` for the 6-item auto-validation, sensitivity sweep, top-100 audits, and submission CSV with `Outlet_ID` + 20,000 rows.
